# Object Detection API Demo

<table align="left"><td>
  <a target="_blank"  href="https://colab.sandbox.google.com/github/tensorflow/models/blob/master/research/object_detection/colab_tutorials/object_detection_tutorial.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab
  </a>
</td><td>
  <a target="_blank"  href="https://github.com/tensorflow/models/blob/master/research/object_detection/colab_tutorials/object_detection_tutorial.ipynb">
    <img width=32px src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td></table>

Welcome to the [Object Detection API](https://github.com/tensorflow/models/tree/master/research/object_detection). This notebook will walk you step by step through the process of using a pre-trained model to detect objects in an image.

> **Important**: This tutorial is to help you through the first step towards using [Object Detection API](https://github.com/tensorflow/models/tree/master/research/object_detection) to build models. If you just just need an off the shelf model that does the job, see the [TFHub object detection example](https://colab.sandbox.google.com/github/tensorflow/docs/blob/master/site/en/hub/tutorials/object_detection.ipynb).

# Setup

Important: If you're running on a local machine, be sure to follow the [installation instructions](https://github.com/tensorflow/models/blob/master/research/object_detection/g3doc/tf2.md). This notebook includes only what's necessary to run in Colab.

### Install

In [ ]:
!pip install -U --pre tensorflow=="2.*"
!pip install tf_slim

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
if True:
    !mkdir "/content/drive/My Drive/tf_od_demo"
%cd "/content/drive/My Drive/tf_od_demo"

Make sure you have `pycocotools` installed

In [ ]:
!pip install pycocotools

Get `tensorflow/models` or `cd` to parent directory of the repository.

In [ ]:
import os
import pathlib


if "models" in pathlib.Path.cwd().parts:
  while "models" in pathlib.Path.cwd().parts:
    os.chdir('..')
elif not pathlib.Path('models').exists():
  !git clone --depth 1 https://github.com/tensorflow/models

Compile protobufs and install the object_detection package

In [ ]:
%%bash
cd models/research/
protoc object_detection/protos/*.proto --python_out=.

In [ ]:
#не запускать
%%bash
cd models/research
pip install .

In [ ]:
#переходим в нужную директорию !1
import os
os.chdir('/content/models/research')

#компилируем protobuf
!protoc object_detection/protos/*.proto --python_out=.

#устанавливаем Object Detection API ДЛЯ TF2
#копируем правильный setup.py для TF2 и устанавливаем
!cp object_detection/packages/tf2/setup.py .
!pip install .

In [ ]:
#запускать это вместо предыдущего
#удаляем конфликтующие пакеты, которые мы пытались установить вручную
!pip uninstall object-detection tf-models-official -y 2>/dev/null || echo "Пакеты не найдены, продолжаем."

#устанавливаем/обновляем совместимый пакет от TensorFlow
!pip install -q --upgrade tf-models-official

print("Установка завершена. Проверяем импорт...")

#проверяем, что теперь импорт работает
import sys
#добавляем пути, которые использует официальный пакет
sys.path.append('/usr/local/lib/python3.12/dist-packages')
sys.path.append('/usr/local/lib/python3.12/dist-packages/object_detection')

try:
    #пробуем импортировать проблемный модуль
    from tensorflow.compat.v1 import estimator
    print("Модуль 'estimator' импортирован успешно.")
    #пробуем импортировать что-то из object_detection
    from object_detection.utils import config_util
    print("Модуль 'object_detection' импортирован успешно.")
except ImportError as e:
    print(f"Ошибка импорта: {e}")

In [ ]:

!pip install -q tensorflow

#клониравать репозиторий и установить Object Detection API
import os
os.chdir('/content')
!git clone -q https://github.com/tensorflow/models
os.chdir('models/research')
!protoc object_detection/protos/*.proto --python_out=.
!cp object_detection/packages/tf2/setup.py .
!pip install -e . --no-deps
print("Object Detection API установлен поверх вашего TensorFlow.")

### Imports

In [ ]:
import numpy as np
import os
import six.moves.urllib as urllib
import sys
import tarfile
import tensorflow as tf
import zipfile

from collections import defaultdict
from io import StringIO
from matplotlib import pyplot as plt
from PIL import Image
from IPython.display import display

In [ ]:
!pip install protobuf==3.20.3 --force-reinstall --no-deps
print("Protobuf понижен до 3.20.3. Необходима перезагрузка среды.")

Import the object detection module.

In [ ]:
from object_detection.utils import ops as utils_ops
from object_detection.utils import label_map_util
from object_detection.utils import visualization_utils as vis_util

Patches:

In [ ]:
# patch tf1 into `utils.ops`


# Patch the location of gfile


# Model preparation

## Variables

Any model exported using the `export_inference_graph.py` tool can be loaded here simply by changing the path.

By default we use an "SSD with Mobilenet" model here. See the [detection model zoo](https://github.com/tensorflow/models/blob/master/research/object_detection/g3doc/detection_model_zoo.md) for a list of other models that can be run out-of-the-box with varying speeds and accuracies.

## Loader

In [ ]:
def load_model(model_name):
  base_url = 'http://download.tensorflow.org/models/object_detection/'
  model_file = model_name + '.tar.gz'
  model_dir = tf.keras.utils.get_file(
    fname=model_name,
    origin=base_url + model_file,
    untar=True)

  model_dir = pathlib.Path(model_dir)/"saved_model"

  model = tf.saved_model.load(str(model_dir))

  return model

## Loading label map
Label maps map indices to category names, so that when our convolution network predicts `5`, we know that this corresponds to `airplane`.  Here we use internal utility functions, but anything that returns a dictionary mapping integers to appropriate string labels would be fine

In [ ]:
# List of the strings that is used to add correct label for each box.
PATH_TO_LABELS = 'models/research/object_detection/data/mscoco_label_map.pbtxt'
category_index = label_map_util.create_category_index_from_labelmap(PATH_TO_LABELS, use_display_name=True)

For the sake of simplicity we will test on 2 images:

In [ ]:
# If you want to test the code with your images, just add path to the images to the TEST_IMAGE_PATHS.
PATH_TO_TEST_IMAGES_DIR = pathlib.Path('models/research/object_detection/test_images')
TEST_IMAGE_PATHS = sorted(list(PATH_TO_TEST_IMAGES_DIR.glob("*.jpg")))
TEST_IMAGE_PATHS

# Detection

Load an object detection model:

In [ ]:
model_name = 'ssd_mobilenet_v1_coco_2017_11_17'
detection_model = load_model(model_name)

Check the model's input signature, it expects a batch of 3-color images of type uint8:

In [ ]:
print(detection_model.signatures['serving_default'].inputs)

And returns several outputs:

In [ ]:
detection_model.signatures['serving_default'].output_dtypes

In [ ]:
detection_model.signatures['serving_default'].output_shapes

Add a wrapper function to call the model, and cleanup the outputs:

In [ ]:
def run_inference_for_single_image(model, image):
  image = np.asarray(image)
  # The input needs to be a tensor, convert it using `tf.convert_to_tensor`.
  input_tensor = tf.convert_to_tensor(image)
  # The model expects a batch of images, so add an axis with `tf.newaxis`.
  input_tensor = input_tensor[tf.newaxis,...]

  # Run inference
  model_fn = model.signatures['serving_default']
  output_dict = model_fn(input_tensor)

  # All outputs are batches tensors.
  # Convert to numpy arrays, and take index [0] to remove the batch dimension.
  # We're only interested in the first num_detections.
  num_detections = int(output_dict.pop('num_detections'))
  output_dict = {key:value[0, :num_detections].numpy()
                 for key,value in output_dict.items()}
  output_dict['num_detections'] = num_detections

  # detection_classes should be ints.
  output_dict['detection_classes'] = output_dict['detection_classes'].astype(np.int64)

  # Handle models with masks:
  if 'detection_masks' in output_dict:
    # Reframe the the bbox mask to the image size.
    detection_masks_reframed = utils_ops.reframe_box_masks_to_image_masks(
              output_dict['detection_masks'], output_dict['detection_boxes'],
               image.shape[0], image.shape[1])
    detection_masks_reframed = tf.cast(detection_masks_reframed > 0.5,
                                       tf.uint8)
    output_dict['detection_masks_reframed'] = detection_masks_reframed.numpy()

  return output_dict

Run it on each test image and show the results:

In [ ]:
def show_inference(model, image_path):
  # the array based representation of the image will be used later in order to prepare the
  # result image with boxes and labels on it.
  image_np = np.array(Image.open(image_path))
  # Actual detection.
  output_dict = run_inference_for_single_image(model, image_np)
  # Visualization of the results of a detection.
  vis_util.visualize_boxes_and_labels_on_image_array(
      image_np,
      output_dict['detection_boxes'],
      output_dict['detection_classes'],
      output_dict['detection_scores'],
      category_index,
      instance_masks=output_dict.get('detection_masks_reframed', None),
      use_normalized_coordinates=True,
      line_thickness=8)

  display(Image.fromarray(image_np))

In [ ]:
for image_path in TEST_IMAGE_PATHS:
  show_inference(detection_model, image_path)


## Instance Segmentation

In [ ]:
model_name = "mask_rcnn_inception_resnet_v2_atrous_coco_2018_01_28"
masking_model = load_model(model_name)

The instance segmentation model includes a `detection_masks` output:

In [ ]:
masking_model.output_shapes

In [ ]:
for image_path in TEST_IMAGE_PATHS:
  show_inference(masking_model, image_path)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "/content/drive/My Drive/tf_od_demo"

In [ ]:
# !2
if True:
    !7z x my_data.7z

In [ ]:
import pandas as pd
import os
from PIL import Image
annot = pd.read_csv('my_data/annot.csv')
annot.head()

In [ ]:
def create_tf_example(example):

    img_fpath = os.path.join('my_data', example.id)
    img = Image.open(img_fpath)
    height = img.size[1]
    width = img.size[0]
    filename = str.encode(example.id)
    with open(img_fpath, mode='rb') as f:
        encoded_image_data = f.read()
    image_format = b'jpeg'

    # List of normalized left x coordinates in bounding box (1 per box)
    xmins = [example.xmin / float(width)]
    # List of normalized right x coordinates in bounding box # (1 per box)
    xmaxs = [example.xmax / float(width)]
    # List of normalized top y coordinates in bounding box (1 per box)
    ymins = [example.ymin / float(height)]
    # List of normalized bottom y coordinates in bounding box # (1 per box)
    ymaxs = [example.ymax / float(height)]
    # List of string class name of bounding box (1 per box)
    classes_text = [b'Cube']
    # List of integer class id of bounding box (1 per box)
    classes = [1]

    tf_example = tf.train.Example(features=tf.train.Features(feature={
        'image/height': dataset_util.int64_feature(height),
        'image/width': dataset_util.int64_feature(width),
        'image/filename': dataset_util.bytes_feature(filename),
        'image/source_id': dataset_util.bytes_feature(filename),
        'image/encoded': dataset_util.bytes_feature(encoded_image_data),
        'image/format': dataset_util.bytes_feature(image_format),
        'image/object/bbox/xmin': dataset_util.float_list_feature(xmins),
        'image/object/bbox/xmax': dataset_util.float_list_feature(xmaxs),
        'image/object/bbox/ymin': dataset_util.float_list_feature(ymins),
        'image/object/bbox/ymax': dataset_util.float_list_feature(ymaxs),
        'image/object/class/text': dataset_util.bytes_list_feature(classes_text),
        'image/object/class/label': dataset_util.int64_list_feature(classes),
    }))
    return tf_example

In [ ]:
import tensorflow as tf
from object_detection.utils import dataset_util
writer = tf.io.TFRecordWriter('my_data/train_data.record')

for idx, row in annot.iterrows():
    tf_example = create_tf_example(row)
    writer.write(tf_example.SerializeToString())

writer.close()

In [ ]:
#скачиваем модель
!wget http://download.tensorflow.org/models/object_detection/tf2/20200711/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8.tar.gz
#распаковываем
!tar -xzf ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8.tar.gz

In [ ]:
from object_detection.utils import label_map_util

In [ ]:
import os

#путь к архиву на вашем Google Диске
archive_on_drive = '/content/drive/My Drive/tf_od_demo/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8.tar.gz'

#копируем архив в рабочую директорию Colab
!cp "{archive_on_drive}" /content/

#переходим в директорию /content и распаковываем архив
os.chdir('/content')
!tar -xzf ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8.tar.gz

#проверяем, что появились правильные файлы
print("Содержимое распакованной папки:")
!ls -la ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/

In [ ]:
model_dir = '/content/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8'
print("Проверяем наличие чекпоинтов...")
#ищем все файлы, связанные с чекпоинтами
!find {model_dir} -name "*.ckpt*" -o -name "checkpoint*" | head -10

In [ ]:
import os
import re

#настройки
#путь файлу конфигурации на Google Диске
config_path = '/content/drive/My Drive/tf_od_demo/my_data/pipeline.config'

#путь к скачанному чекпоинту модели
checkpoint_path = '/content/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/checkpoint/ckpt-0'

#путь к  label_map.pbtxt
label_map_path = '/content/drive/My Drive/tf_od_demo/my_data/cube_label_map.pbtxt'

#пути к .record файлам
train_record_path = '/content/drive/My Drive/tf_od_demo/my_data/train_data.record'
eval_record_path = '/content/drive/My Drive/tf_od_demo/my_data/train_data.record'

#количество классов
num_classes = 1

In [ ]:
import shutil
import os

#актуальный конфиг в библиотеке
source_config = '/content/models/research/object_detection/configs/tf2/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8.config'

#путь, куда нужно скопировать конфиг
destination_config = '/content/drive/My Drive/tf_od_demo/my_data/pipeline.config'

#копируем, заменяя старый файл
shutil.copyfile(source_config, destination_config)
print(f"Актуальный конфиг скопирован и заменил старый файл: {destination_config}")

#для проверки первые несколько строк нового файла
print("\nПервые 10 строк нового конфига:")
!head -10 '/content/drive/My Drive/tf_od_demo/my_data/pipeline.config'

In [ ]:
#создание файла label_map.pbtxt
label_map_content = """
item {
  id: 1
  name: 'cube'
}
"""
label_map_path = '/content/drive/My Drive/tf_od_demo/my_data/cube_label_map.pbtxt'
with open(label_map_path, 'w') as f:
    f.write(label_map_content)
print(f"Файл label_map создан: {label_map_path}")
!cat '{label_map_path}'

In [ ]:
import os
record_path = '/content/drive/My Drive/tf_od_demo/my_data/train_data.record'
if os.path.exists(record_path):
    print(f"Файл .record найден. Размер: {os.path.getsize(record_path)} байт")

In [ ]:
#НЕ ВЫПОЛНЯТЬ ЯЧЕЙКУ

#переустанавливаем protobuf в совместимую версию 3.20.3
!pip install protobuf==3.20.3 --force-reinstall

print("Protobuf downgraded. The environment must be restarted...")

#перезагружаем среду выполнения Colab для применения изменений
import os
os.kill(os.getpid(), 9)

In [ ]:
# !3
%%time
import os

#определяем пути
pipeline_config = '/content/drive/My Drive/tf_od_demo/my_data/pipeline.config'
model_output_dir = '/content/drive/My Drive/tf_od_demo/my_data/output'

#создаем папку для результатов
os.makedirs(model_output_dir, exist_ok=True)

#используем model_main_tf2.py и полные пути
!cd /content/models/research && python object_detection/model_main_tf2.py \
    --pipeline_config_path="{pipeline_config}" \
    --model_dir="{model_output_dir}" \
    --num_train_steps=1000 \
    --alsologtostderr

In [ ]:
#обновляем списки пакетов и устанавливаем Python 3.9 и модуль venv
!apt-get update -qq
!apt-get install -y python3.9 python3.9-venv python3.9-dev

#проверяем установку
!python3.9 --version

print("\n Python 3.9 установлен.")

In [ ]:
!rm -rf /content/tf_od_venv_py39

#создаём новое виртуальное окружение
!python3.9 -m venv /content/tf_od_venv_py39

#проверяем, что окружение создалось
print("\nПроверяем содержимое папки /content/tf_od_venv_py39/bin:")
!ls -la /content/tf_od_venv_py39/bin/python*

print("\n Виртуальное окружение создано.")

In [ ]:
pip_path_39 = '/content/tf_od_venv_py39/bin/pip'
python_path_39 = '/content/tf_od_venv_py39/bin/python'

#обновляем pip внутри окружения (важно для совместимости)
!{pip_path_39} install --upgrade pip

#встанавливаем TensorFlow 2.13.0
!{pip_path_39} install tensorflow==2.13.0

#фиксируем версию protobuf
!{pip_path_39} install "protobuf<3.20"

#устанавливаем остальные зависимости
!{pip_path_39} install matplotlib pandas pillow lxml Cython contextlib2 tf_slim

#проверяем установку
!{python_path_39} -c "import tensorflow as tf; print(' TensorFlow версия:', tf.__version__)"

print("\n Все пакеты установлены.")

In [ ]:
if not os.path.exists('/content/models'):
    !git clone -q https://github.com/tensorflow/models

# компилируем .proto файлы с protoc из виртуального окружения
!cd /content/models/research && {python_path_39} -m grpc_tools.protoc object_detection/protos/*.proto --python_out=.

#устанавливаем object_detection БЕЗ зависимостей
!cd /content/models/research && {pip_path_39} install . --no-deps

#устанавливаем совместимую версию tf-models-official
!{pip_path_39} install tf-models-official==2.13.0

print("\n API установлен.")

In [ ]:
import os

#важные настройки
os.environ['MPLBACKEND'] = 'Agg'
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

#запускаем обучение из виртуального окружения Python 3.9
train_cmd = f'''
cd /content/models/research && \
{python_path_39} object_detection/model_main_tf2.py \
    --pipeline_config_path="/content/drive/My Drive/tf_od_demo/my_data/pipeline.config" \
    --model_dir="/content/drive/My Drive/tf_od_demo/my_data/output" \
    --num_train_steps=1000 \
    --alsologtostderr
'''
print("Запускаем обучение")
!{train_cmd}

In [ ]:
#проверяем, что пакет установлен
!/content/tf_od_venv_py39/bin/pip list | grep object-detection

#ищем где находится модуль object_detection
!/content/tf_od_venv_py39/bin/python -c "import sys; print('\n'.join(sys.path))" | grep -i object

In [ ]:
import sys
#добавляем путь к research и object_detection в sys.path
sys.path.append('/content/models/research')
sys.path.append('/content/models/research/slim')  # tf-slim тоже нужен

#проверяем
try:
    import object_detection
    print(' Модуль object_detection импортирован успешно')
except Exception as e:
    print(f' Ошибка: {e}')

In [ ]:
#проверяем системный protoc
!which protoc
!protoc --version

#проверяем protoc в виртуальном окружении
!/content/tf_od_venv_py39/bin/protoc --version

#если в виртуальном окружении нет protoc, устанавливаем
try:
    !/content/tf_od_venv_py39/bin/protoc --version
except:
    print("Устанавливаем protoc в виртуальное окружение...")
    !/content/tf_od_venv_py39/bin/pip install protobuf

In [ ]:
import os
os.chdir('/content/models/research')

# Компилируем ВСЕ .proto файлы
!protoc object_detection/protos/*.proto --python_out=.

# Проверяем результат
print("\n Проверяем созданные файлы:")
proto_files = !ls object_detection/protos/*_pb2.py 2>/dev/null || true
print(f"Найдено {len(proto_files)} *_pb2.py файлов")

if len(proto_files) > 0:
    print("Первые 5 файлов:")
    for f in proto_files[:5]:
        print(f"  - {f}")
else:
    print(" Файлы не создались!")

In [ ]:
!/content/tf_od_venv_py39/bin/pip install --upgrade "pip<24.1"

# Проверяем версию
!/content/tf_od_venv_py39/bin/pip --version

In [ ]:
# Устанавливаем lvis (используем совместимую версию)
!/content/tf_od_venv_py39/bin/pip install lvis

# Проверяем установку
!/content/tf_od_venv_py39/bin/python -c "import lvis; print(' LVIS установлен:', lvis.__version__)"

In [ ]:
!/content/tf_od_venv_py39/bin/pip uninstall -y numpy

# 2. Устанавливаем numpy 1.24.3 (совместимый с TensorFlow 2.13.0)
!/content/tf_od_venv_py39/bin/pip install numpy==1.24.3

# 3. Проверяем
!/content/tf_od_venv_py39/bin/python -c "import numpy as np; print(f' NumPy версия: {np.__version__}')"

In [ ]:
!/content/tf_od_venv_py39/bin/pip list | grep -E "(tensorflow|numpy|lvis|opencv)"

In [ ]:
!/content/tf_od_venv_py39/bin/pip install tensorflow-io

In [ ]:
!/content/tf_od_venv_py39/bin/pip uninstall tensorflow tensorflow-cpu -y
!/content/tf_od_venv_py39/bin/pip install "tensorflow[and-cuda]==2.13.0"

In [ ]:
!/content/tf_od_venv_py39/bin/python -c "import tensorflow as tf; print('Версия в ВО:', tf.__version__); print('Доступны ли GPU:', tf.config.list_physical_devices('GPU'))"

In [ ]:
# Установка CUDA для виртуального окружения в Colab
!source /content/tf_od_venv_py39/bin/activate && \
wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/cuda-keyring_1.1-1_all.deb && \
dpkg -i cuda-keyring_1.1-1_all.deb && \
apt-get update && \
apt-get install -y cuda-toolkit-11-8 --no-install-recommends

!source /content/tf_od_venv_py39/bin/activate && \
wget -q https://developer.download.nvidia.com/compute/redist/cudnn/v8.6.0/local_installers/11.8/cudnn-linux-x86_64-8.6.0.163_cuda11-archive.tar.xz && \
tar -xf cudnn-linux-x86_64-8.6.0.163_cuda11-archive.tar.xz && \
cp -r cudnn-*-archive/include/* /content/tf_od_venv_py39/include/ 2>/dev/null || true && \
cp -r cudnn-*-archive/lib/* /content/tf_od_venv_py39/lib/ 2>/dev/null || true

# Настройка переменных окружения
!echo 'export LD_LIBRARY_PATH=/content/tf_od_venv_py39/lib:$LD_LIBRARY_PATH' >> /content/tf_od_venv_py39/bin/activate
!echo 'export XLA_FLAGS=--xla_gpu_cuda_data_dir=/usr/local/cuda-11.8' >> /content/tf_od_venv_py39/bin/activate

print("Библиотеки установлены! Теперь:")
print("1. Runtime → Restart runtime")
print("2. После перезапуска проверьте GPU командой ниже")
print("3. Запустите обучение как обычно")

In [ ]:
!/content/tf_od_venv_py39/bin/python -c "import tensorflow as tf; print('GPU доступны:', tf.config.list_physical_devices('GPU'))"

In [ ]:
#копируем ВСЕ необходимые библиотеки CUDA 11.8 в окружение
!mkdir -p /content/tf_od_venv_py39/lib
!cp /usr/local/cuda-11.8/lib64/libcudart* /content/tf_od_venv_py39/lib/
!cp /usr/local/cuda-11.8/lib64/libcublas* /content/tf_od_venv_py39/lib/
!cp /usr/local/cuda-11.8/lib64/libcufft* /content/tf_od_venv_py39/lib/
!cp /usr/local/cuda-11.8/lib64/libcurand* /content/tf_od_venv_py39/lib/
!cp /usr/local/cuda-11.8/lib64/libcusolver* /content/tf_od_venv_py39/lib/
!cp /usr/local/cuda-11.8/lib64/libcusparse* /content/tf_od_venv_py39/lib/
!cp /usr/local/cuda-11.8/lib64/libcudnn* /content/tf_od_venv_py39/lib/

#проверяем, что скопировалось
!ls /content/tf_od_venv_py39/lib/*.so* | head -20

In [ ]:
import os

# Пути к файлам
config_path = "/content/drive/My Drive/tf_od_demo/my_data/pipeline.config"
data_dir = os.path.dirname(config_path)  # Папка с данными

# Ваши реальные файлы
label_map_file = "cube_label_map.pbtxt"  # Ваш файл с метками
train_record_file = "train_data.record"   # Ваш тренировочный файл

# Полные пути
label_map_path = os.path.join(data_dir, label_map_file)
train_record_path = os.path.join(data_dir, train_record_file)

print(f"Папка с данными: {data_dir}")
print(f"1. Файл с метками: {label_map_path}")
print(f"2. Тренировочный файл: {train_record_path}")

# Проверяем, что файлы существуют
if os.path.exists(label_map_path):
    print(f" {label_map_file} найден")
else:
    print(f" {label_map_file} НЕ найден! Проверьте путь.")

if os.path.exists(train_record_path):
    print(f" {train_record_file} найден")
else:
    print(f" {train_record_file} НЕ найден! Проверьте путь.")

# Читаем конфиг
with open(config_path, 'r') as f:
    content = f.read()

print("\n=== ЗАМЕНЯЕМ ПУТИ ===")

# 1. Заменяем label_map_path
old_label = 'label_map_path: "PATH_TO_BE_CONFIGURED/label_map.txt"'
new_label = f'label_map_path: "{label_map_path}"'
content = content.replace(old_label, new_label)
print(f"1. Исправлен label_map_path: {new_label}")

# 2. Заменяем input_path для train
old_train = 'input_path: "PATH_TO_BE_CONFIGURED/train2017-?????-of-00256.tfrecord"'
new_train = f'input_path: "{train_record_path}"'
content = content.replace(old_train, new_train)
print(f"2. Исправлен train input_path: {new_train}")

# 3. Если есть eval_input_reader, нужно найти eval файл
# Сначала поищем eval/val/test файлы
eval_files = []
for f in os.listdir(data_dir):
    if f.endswith('.record') and ('val' in f.lower() or 'eval' in f.lower() or 'test' in f.lower()):
        eval_files.append(os.path.join(data_dir, f))

if 'eval_input_reader' in content:
    print("3. Найден eval_input_reader в конфиге")

    if eval_files:
        eval_record_path = eval_files[0]  # Берём первый подходящий
        old_eval = 'input_path: "PATH_TO_BE_CONFIGURED/val2017-?????-of-00032.tfrecord"'
        new_eval = f'input_path: "{eval_record_path}"'
        content = content.replace(old_eval, new_eval)
        print(f"   Исправлен eval input_path: {new_eval}")
    else:
        print("     Не найден eval/val/test.record файл!")
        print("   Если у вас нет отдельного eval файла, используйте тот же train файл или создайте его.")
        # Можно временно использовать train файл для eval
        old_eval = 'input_path: "PATH_TO_BE_CONFIGURED/val2017-?????-of-00032.tfrecord"'
        new_eval = f'input_path: "{train_record_path}"'  # Используем train файл
        content = content.replace(old_eval, new_eval)
        print(f"   Временно используем train файл для eval: {new_eval}")

# 4. Сохраняем исправленный конфиг
backup_path = config_path + ".backup"
!cp "{config_path}" "{backup_path}"
print(f"\n Создана резервная копия: {backup_path}")

with open(config_path, 'w') as f:
    f.write(content)

print(" Конфиг обновлён!")

# 5. Показываем исправленные строки
print("\n=== ПРОВЕРКА ===")
print("Исправленные строки в конфиге:")
!grep -n "label_map_path\|input_path" "{config_path}"

In [ ]:
# Правильный путь к checkpoint
checkpoint_prefix = "/content/drive/My Drive/tf_od_demo/my_data/checkpoint/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/checkpoint/ckpt-0"
print(f"Правильный путь к checkpoint: {checkpoint_prefix}")

# Проверяем, что файлы существуют
print("\nПроверяем существование файлов:")
!ls -la "{checkpoint_prefix}"*


In [ ]:
import re
# Обновляем конфиг
config_path = "/content/drive/My Drive/tf_od_demo/my_data/pipeline.config"

with open(config_path, 'r') as f:
    content = f.read()

# Заменяем путь
old_patterns = [
    r'fine_tune_checkpoint: "PATH_TO_BE_CONFIGURED/[^"]+"',
    r'fine_tune_checkpoint: "[^"]*mobilenet[^"]*"',
    r'fine_tune_checkpoint: "[^"]*ckpt[^"]*"'
]

new_checkpoint_line = f'fine_tune_checkpoint: "{checkpoint_prefix}"'

for pattern in old_patterns:
    matches = re.findall(pattern, content)
    for match in matches:
        print(f"Найдена строка для замены: {match}")
        content = content.replace(match, new_checkpoint_line)

# Сохраняем
with open(config_path, 'w') as f:
    f.write(content)

In [ ]:
print("\nПроверяем исправленные строки в конфиге:")
!grep -n "fine_tune_checkpoint" "{config_path}"

In [ ]:
output_dir = "/content/drive/My Drive/tf_od_demo/my_data/output"

if os.path.exists(output_dir):
    print(f"Содержимое папки {output_dir}:")
    !ls -la "{output_dir}/"

    # Проверяем чекпоинты
    checkpoint_files = !find "{output_dir}" -name "*.index" 2>/dev/null || true

    if checkpoint_files:
        print("\nНайдены чекпоинты (обучение сохранялось):")
        for f in checkpoint_files:
            print(f"  - {f}")

        # Определяем последний шаг
        import re
        steps = []
        for f in checkpoint_files:
            match = re.search(r'ckpt-(\d+)\.index', f)
            if match:
                steps.append(int(match.group(1)))

        if steps:
            max_step = max(steps)
            print(f"\n📊 Максимальный выполненный шаг: {max_step}")
            print(f"   Осталось шагов: {1000 - max_step}")
    else:
        print("Чекпоинтов не найдено (обучение не сохранилось)")
else:
    print(f"Папка {output_dir} не существует")

In [ ]:
#предотвращаем отключение Colab
from IPython.display import Javascript
Javascript("""
function keepAlive() {
    console.log("Keeping Colab alive...");
}
setInterval(keepAlive, 60000);
""")

In [ ]:
import re

config_path = "/content/drive/My Drive/tf_od_demo/my_data/pipeline.config"

with open(config_path, 'r') as f:
    content = f.read()

# Ищем batch_size
batch_match = re.search(r'batch_size:\s*(\d+)', content)
if batch_match:
    print(f"batch_size в конфиге: {batch_match.group(1)}")

    # Если больше 8, уменьшаем
    current_batch = int(batch_match.group(1))
    if current_batch > 8:
        new_batch = 4  # Сильно уменьшаем для теста
        content = re.sub(r'batch_size:\s*\d+', f'batch_size: {new_batch}', content)

        with open(config_path, 'w') as f:
            f.write(content)

        print(f"batch_size уменьшен: {current_batch} → {new_batch}")
    else:
        print(f"batch_size уже маленький ({current_batch})")
else:
    print("batch_size не найден в конфиге")

    # Добавляем batch_size если его нет
    content = content.replace('train_config {', 'train_config {\n  batch_size: 4')
    with open(config_path, 'w') as f:
        f.write(content)
    print("Добавлен batch_size: 4")

In [ ]:


import re

config_path = "/content/drive/My Drive/tf_od_demo/my_data/pipeline.config"

#читаем оригинальный конфиг из скачанных весов
original_config = "/content/ssd_mobilenet_v2_fpnlite_640x640_coco17_tpu-8/pipeline.config"

with open(original_config, 'r') as f:
    original_content = f.read()

#читаем текущий конфиг (чтобы сохранить ваши пути)
with open(config_path, 'r') as f:
    current_content = f.read()

#находим и заменяем только image_resizer
#ищем image_resizer в оригинальном конфиге
image_resizer_match = re.search(r'image_resizer \{[^}]+\}', original_content, re.DOTALL)
if image_resizer_match:
    original_resizer = image_resizer_match.group(0)
    print("Найден оригинальный image_resizer")

    # Заменяем image_resizer в текущем конфиге
    current_content = re.sub(r'image_resizer \{[^}]+\}', original_resizer, current_content, flags=re.DOTALL)

    with open(config_path, 'w') as f:
        f.write(current_content)

    print("Размер изображений восстановлен до оригинального")
else:
    print("Не найден image_resizer в оригинальном конфиге")

#устанавливаем batch_size: 16 (или 8 если мало памяти)
print("\nустанавливаем BATCH_SIZE: 16 ")

# Находим и устанавливаем batch_size
if 'batch_size:' in current_content:
    # Заменяем текущий batch_size
    current_content = re.sub(r'batch_size:\s*\d+', 'batch_size: 16', current_content)
else:
    # Добавляем batch_size если его нет
    current_content = current_content.replace('train_config {', 'train_config {\n  batch_size: 16')

with open(config_path, 'w') as f:
    f.write(current_content)

print("batch_size установлен: 16")

#проверяем изменения
print("\nроверка конфига")
!grep -n "height\|width\|batch_size" "{config_path}"

In [ ]:
import os

os.environ['MPLBACKEND'] = 'Agg'
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

config_path = "/content/drive/My Drive/tf_od_demo/my_data/pipeline.config"

# 1. Читаем конфиг
with open(config_path, 'r') as f:
    lines = f.readlines()

# 2. Ищем и исправляем проблему
fixed_lines = []
for line in lines:
    # Удаляем пустую строку где было fine_tune_checkpoint
    if 'fine_tune_checkpoint:' in line:
        # Пропускаем эту строку (мы её удалили ранее)
        continue
    # Также проверяем, нет ли пустых строк с отступами
    if line.strip() == '' and '  ' in line:  # Пустая строка с отступами
        continue
    fixed_lines.append(line)

# 3. Сохраняем исправленный конфиг
with open(config_path, 'w') as f:
    f.writelines(fixed_lines)

print("Конфиг очищен от пустых строк")

# 4. Проверяем конфиг
print("\nПроверяем конфиг (первые 30 строк):")
!head -30 "{config_path}"

In [ ]:
import re

config_path = "/content/drive/My Drive/tf_od_demo/my_data/pipeline.config"

with open(config_path, 'r') as f:
    content = f.read()

# Уменьшаем batch_size до 1
content = re.sub(r'batch_size:\s*\d+', 'batch_size: 1', content)

with open(config_path, 'w') as f:
    f.write(content)

print("batch_size уменьшен до 1")

In [ ]:
#убедимся, что поменялось правильно
with open(config_path, 'r') as f:
    content = f.read()

print("Проверка изменений:")
print(" " * 40)

#ищем все упоминания batch_size
for line in content.split('\n'):
    if 'batch_size' in line:
        print(line.strip())

In [ ]:
!grep -A 10 "data_augmentation_options" "/content/drive/My Drive/tf_od_demo/my_data/pipeline.config"

In [ ]:
#полная секция optimizer
config_path = "/content/drive/My Drive/tf_od_demo/my_data/pipeline.config"

with open(config_path, 'r') as f:
    content = f.read()

#нахождение секции optimizer
import re
optimizer_section = re.search(r'optimizer\s*\{[^}]+\}', content, re.DOTALL)
if optimizer_section:
    print("Ваш оптимизатор:")
    print(optimizer_section.group(0))
else:
    print("Не найден optimizer в конфиге")

In [ ]:
import os

# Критически важные настройки для GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['MPLBACKEND'] = 'Agg'

# Проверяем, что TensorFlow видит GPU
import tensorflow as tf
print("Доступные GPU устройства:", tf.config.list_physical_devices('GPU'))

# Если GPU не виден, возможно нужна другая версия TF
if not tf.config.list_physical_devices('GPU'):
    print("TensorFlow не видит GPU!")
    print("Попробуйте переустановить TensorFlow с GPU поддержкой:")
    print("!/content/tf_od_venv_py39/bin/pip install tensorflow[and-cuda]==2.13.0")

In [ ]:
#не нужно
#!ls /usr/local/cuda*

In [ ]:
#создаем символические ссылки в виртуальном окружении
!mkdir -p /content/tf_od_venv_py39/lib
!ln -sf /usr/local/cuda-11.8/lib64/libcudart.so.11.0 /content/tf_od_venv_py39/lib/
!ln -sf /usr/local/cuda-11.8/lib64/libcudart.so /content/tf_od_venv_py39/lib/

In [ ]:
train_cmd = '''
export LD_LIBRARY_PATH=/content/tf_od_venv_py39/lib:/usr/local/cuda-11.8/lib64:$LD_LIBRARY_PATH
export XLA_FLAGS=--xla_gpu_cuda_data_dir=/usr/local/cuda-11.8
export CUDA_VISIBLE_DEVICES=0

cd /content/models/research && \
PYTHONPATH=/content/models/research:/content/models/research/slim:$PYTHONPATH \
/content/tf_od_venv_py39/bin/python object_detection/model_main_tf2.py \
    --pipeline_config_path="/content/drive/My Drive/tf_od_demo/my_data/pipeline.config" \
    --model_dir="/content/drive/My Drive/tf_od_demo/my_data/output" \
    --num_train_steps=10000 \
    --checkpoint_every_n=1000 \
    --alsologtostderr
'''

print("Запускаем обучение на 10000 шагов...")

!{train_cmd}

In [ ]:
!source /content/tf_od_venv_py39/bin/activate && cd /content/models/research && PYTHONPATH=/content/models/research:/content/models/research/slim:$PYTHONPATH python object_detection/exporter_main_v2.py --input_type=image_tensor --pipeline_config_path="/content/drive/My Drive/tf_od_demo/my_data/pipeline.config" --trained_checkpoint_dir="/content/drive/My Drive/tf_od_demo/my_data/output" --output_directory="/content/drive/My Drive/tf_od_demo/my_data/frozen"

In [ ]:
#проверка после экспорта
import os

frozen_path = '/content/drive/My Drive/tf_od_demo/my_data/frozen'
print("Содержимое папки frozen:")
!ls -la "$frozen_path" 2>/dev/null || echo "Папки ещё нет"


if os.path.exists(frozen_path):
    print("\nДетально:")
    !find "$frozen_path" -type f -name "*.pb" -o -name "*.pbtxt"

In [ ]:
# Копируем все содержимое папки с виртуальным окружением на Google Диск
!cp -r /content/tf_od_venv_py39 /content/drive/My\ Drive/tf_od_demo/

In [ ]:
#мнтируем Google Диск
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import sys
import os

#добавляем пути к object_detection API
sys.path.insert(0, '/content/models/research')
sys.path.insert(0, '/content/models/research/slim')

from object_detection.utils import label_map_util
from object_detection.utils import visualization_utils as vis_util

In [ ]:
#загружаем модель
print("1. Загружаем модель...")
detect_fn = tf.saved_model.load('/content/drive/My Drive/tf_od_demo/my_data/frozen/saved_model')
print("Модель загружена")

In [ ]:
#згружаем label map
print("2. Загружаем label map...")
category_index = label_map_util.create_category_index_from_labelmap(
    '/content/drive/My Drive/tf_od_demo/my_data/cube_label_map.pbtxt',
    use_display_name=True
)
print(f"Категории: {category_index}")

In [ ]:
#функция детекции для TF2
def detect_objects(image_np, model):
    """Детекция объектов на одном изображении"""
    #конвертируем в тензор TF
    input_tensor = tf.convert_to_tensor(image_np)
    #добавляем batch dimension (1, height, width, 3)
    input_tensor = input_tensor[tf.newaxis, ...]

    #выполняем детекцию
    detections = model(input_tensor)

    #обрабатываем результаты
    num_detections = int(detections.pop('num_detections'))

    #конвертируем в numpy
    result = {}
    for key, value in detections.items():
        result[key] = value[0, :num_detections].numpy()

    result['num_detections'] = num_detections
    result['detection_classes'] = result['detection_classes'].astype(np.int64)

    return result

In [ ]:
#загружаем тестовое изображение
image_path = '/content/drive/My Drive/tf_od_demo/my_data/test.jpg'
print(f"\n3. Загружаем изображение: {image_path}")

if not os.path.exists(image_path):
    print(f" Файл не найден! Проверьте путь.")
    #что есть
    !ls -la "/content/drive/My Drive/tf_od_demo/my_data/"*.jpg 2>/dev/null
else:
    #загружаем изображение
    image_np = imageio.imread(image_path)
    print(f"Изображение загружено. Размер: {image_np.shape}")

    #запускаем детекцию

    output_dict = detect_objects(image_np, detect_fn)
    print(f"Детекция завершена. Всего обнаружений: {output_dict['num_detections']}")

    #пнализируем результаты

    confidence_threshold = 0.3

    #сколько уверенных детекций
    confident_mask = output_dict['detection_scores'] > confidence_threshold
    confident_count = np.sum(confident_mask)

    print(f" Порог уверенности: {confidence_threshold}")
    print(f" Уверенных детекций: {confident_count}")

    if confident_count > 0:
        print(f"Оценки уверенности: {output_dict['detection_scores'][confident_mask][:3]}")
    else:
        print("Нет уверенных детекций.Нужно уменьшить порог.")

    #визуализация

    #создаем копию изображения для рисования
    image_with_boxes = image_np.copy()

    #рисуем bounding boxes
    vis_util.visualize_boxes_and_labels_on_image_array(
        image_with_boxes,
        output_dict['detection_boxes'],
        output_dict['detection_classes'],
        output_dict['detection_scores'],
        category_index,
        use_normalized_coordinates=True,
        line_thickness=6,
        min_score_thresh=confidence_threshold)

    #отображаем результат
    plt.figure(figsize=(16, 12))
    plt.imshow(image_with_boxes)
    plt.axis('off')

    if confident_count > 0:
        plt.title(f'Обнаружено объектов: {confident_count} (порог: {confidence_threshold})',
                  fontsize=16, color='green')
    else:
        plt.title(f'Объекты не обнаружены (порог: {confidence_threshold})',
                  fontsize=16, color='red')

    plt.tight_layout()
    plt.show()

    import matplotlib
    matplotlib.use('Agg')
    %matplotlib inline


    plt.figure(figsize=(16, 12))
    plt.imshow(image_with_boxes)
    plt.axis('off')
    plt.title(f'Найдено. Уверенность: {output_dict["detection_scores"][confident_mask][0]:.1%}',
              fontsize=18, color='green')
    plt.show()
